In [ ]:
import numpy as np
from scipy.stats import poisson
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, hinge_loss, log_loss, brier_score_loss
from sklearn.base import BaseEstimator
from sklearn.svm import SVC
from sklearn.datasets import fetch_openml

from customkernels import MeanOverlapKernel, Kernel1Full, Kernel2Full
from ucimlrepo import fetch_ucirepo
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import calibration_curve


In [ ]:
def GridSearch_k1_k3(X_train, y_train, cont_cols):

    gammas = [2**e for e in range(-3, 3)]          
    alphas = [0.1, 0.2, 0.3, 0.5, 0.7, 0.9, 1.0, 1.5]
    Cs = [0.1, 1, 10]

    # Initialize variables to track the best model
    best_accuracy = 0
    best_params = {}
    best_model = None

    # Add cross-validation (5-fold)
    kf = KFold(n_splits=5, shuffle=True, random_state=42)

    for γ in gammas:
        print(f"Testing gamma={γ}")
        for α in alphas:
            for C in Cs:
                accuracies = []
                
                # Cross-validation loop
                for train_idx, val_idx in kf.split(X_train):
                    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
                    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
                    
                    # Train kernel and SVC
                    k1 = Kernel1Full(
                        continuous_vars=cont_cols,
                        alpha=α,
                        rbf_gamma=γ,
                        cat_gamma=γ
                    ).fit(X_tr, y_tr)
                    
                    svc1 = SVC(kernel=k1, probability=True, C=C)
                    svc1.fit(X_tr, y_tr)
                    y_pred = svc1.predict(X_val)
                    accuracies.append(accuracy_score(y_val, y_pred))
                
                # Average accuracy across folds
                mean_accuracy = np.mean(accuracies)
                
                # Update best model if current is better
                if mean_accuracy > best_accuracy:
                    best_accuracy = mean_accuracy
                    best_params = {'gamma': γ, 'alpha': α, 'C': C}
                    best_model = svc1  

    print("Best accuracy:", best_accuracy)
    print("Best params:", best_params)
    
    return best_model, best_params, best_accuracy

In [ ]:
# same code used for each dataset, just change the path

# 1. dmean dataset
# df = pd.read_csv("dmean_df.csv")
# y = df["ExtractionFlag"]
# X = df.drop(columns=["ExtractionFlag"])

# 2. dmax dataset
# df = pd.read_csv("dmax_df.csv")
# y = df["ExtractionFlag"]
# X = df.drop(columns=["ExtractionFlag"])

# cont_cols = ["TotalDose"]   
# cat_cols = X.drop(columns=['TotalDose']).columns.tolist()

# #  this splitting procedure is only for extraction datasets
# unique_patients = df['PatientID'].unique()

# train_patients, test_patients = train_test_split(
#     unique_patients, 
#     test_size=0.2, 
#     random_state=42
# )

# train_mask = df['PatientID'].isin(train_patients)
# test_mask = df['PatientID'].isin(test_patients)

# X_train = X[train_mask]
# X_test = X[test_mask]
# y_train = y[train_mask]
# y_test = y[test_mask]

# X_train = X_train.drop(columns=["PatientID"])
# X_test = X_test.drop(columns=["PatientID"])

# print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

#  --------------------

# df 3: 
congressional_voting_records = fetch_ucirepo(id=105) 
X = congressional_voting_records.data.features 
y = congressional_voting_records.data.targets 
y = y.iloc[:, 0] 
y = y.map({'republican': 0, 'democrat': 1})
cat_cols = X.columns.tolist()   
cont_cols = []  # No continuous columns in this dataset

# df 4: MONK dataset
# monk_s_problems = fetch_ucirepo(id=70) 
# X = monk_s_problems.data.features 
# y = monk_s_problems.data.targets 
# y = y.map({'republican': 0, 'democrat': 1})
# cat_cols = X.columns.tolist()   
# cont_cols = []  # No continuous columns in this dataset

# this split is used for the UCI datasets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
    )
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In [ ]:
best_model_k1, best_params_k1, best_acc_k1 = GridSearch_k1_k3(X_train, y_train, cont_cols)

In [ ]:
# Train the final kernel using the optimal params on the training set
final_kernel = Kernel1Full(
    continuous_vars=cont_cols,
    alpha=best_params_k1['alpha'],
    rbf_gamma=best_params_k1['gamma'],
    cat_gamma=best_params_k1['gamma']
).fit(X_train, y_train)

svc_final = SVC(
    kernel=final_kernel,
    C=best_params_k1['C'], 
    probability=True
).fit(X_train, y_train)

y_test_pred = svc_final.predict(X_test)

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("Classification Report:")
print(classification_report(y_test, y_test_pred))   

In [ ]:
# get predicted probabilities for the positive class
y_test_proba = svc_final.predict_proba(X_test)[:, 1]

# confusion matrix and classification report with threshold 0.5
y_test_pred = (y_test_proba >= 0.5).astype(int)
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred))
print("Classification Report:")
print(classification_report(y_test, y_test_pred))   

# Brier score (mean squared error of the probabilities)
brier = brier_score_loss(y_test, y_test_proba)
print(f"Brier score: {brier:.4f}")


# Weighted cross-entropy (log-loss with class weights)
classes = np.unique(y_test)
cw = compute_class_weight(class_weight='balanced', classes=classes, y=y_test)
class_weight_dict = dict(zip(classes, cw))

sample_weights = np.array([class_weight_dict[y] for y in y_test])

wlog = log_loss(y_test, y_test_proba, sample_weight=sample_weights)
print(f"Weighted log-loss: {wlog:.4f}")

In [ ]:
# create calibration curve
prob_true, prob_pred = calibration_curve(y_test, y_test_proba, n_bins=10)   

plt.figure(figsize=(8, 6))
plt.plot(prob_pred, prob_true, marker='o', label='Calibration Curve')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Perfectly Calibrated')
plt.xlabel('Predicted Probability')
plt.ylabel('True Probability')
plt.legend()
plt.show()

In [ ]:
# Create bins
bins = np.arange(0, 1.1, 0.1)
bin_labels = [f"{b:.2f}-{b+0.1:.2f}" for b in bins[:-1]]

# Bin the probabilities
df = pd.DataFrame({
    'prob': y_test_proba,
    'true': y_test,
    'bin': pd.cut(y_test_proba, bins=bins, labels=bin_labels, include_lowest=True)
})

# Count correct and wrong predictions in each bin
counts = df.groupby('bin')['true'].value_counts().unstack().fillna(0)
counts.columns = ['Class 0', 'Class 1']

# Plot on a narrower figure
fig, ax = plt.subplots(figsize=(10, 6))   
counts.plot(kind='bar', 
            stacked=True, 
            ax=ax, 
            width=0.75)                   
plt.xlabel('Predicted Probability Bin')
plt.ylabel('Count of Predictions')
plt.xticks(rotation=0)
plt.legend(title='Actual Class', loc='upper right')
plt.tight_layout()

# Add counts on top of bars
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        x, y = p.get_xy() 
        ax.annotate(f'{int(height)}', 
                    (x + p.get_width()/2, y + height/2), 
                    ha='center', va='center', 
                    color='black', fontsize=9)

plt.show()